In [ ]:
# Paths come from `paths.py`, never from a literal relative to some checkout.
# That module is the only place that knows where the tree lives, and it names
# the environment variables that override each location (FSCORE_DB above all —
# fscore.db is ~1.7 GB and is not kept in the repository).
# Run this notebook from its own directory, src/fscore_vietnam.
import sys, pathlib

HERE = pathlib.Path.cwd()
assert (HERE / "paths.py").exists(), f"run from src/fscore_vietnam (cwd={HERE})"
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from paths import DATA, RESULTS, DB, ensure_dirs, require_db
ensure_dirs()

# `run_grid` export

The bridge between this branch and the experiment branch. `price_matching_and_finalize`
ends with `final_panel.csv` — one row per `(symbol, period)` carrying everything the
preprocessing pipeline knows about a Vietnamese firm-year. `run_grid` on `origin/main`
does not read that panel; it reads three normalised inputs, and this notebook is the only
place the two schemas are allowed to meet.

Nothing is recomputed here. Every number that leaves this notebook was already decided
upstream — the F-score in `f_score_calculation`, the book-to-market in
`book_to_market_calculation`, the liquidity gate in `tradable_by_turnover_filter`. What
this notebook does is **rename, gate, and reshape**, and then assert that the result
actually satisfies the contract rather than merely resembling it.

## The three inputs

| input | what it is | written to |
|---|---|---|
| `scores` | one row per `(score_year, ticker)`: F-score, its nine flags, B/M, sector | `vietnam_scores.csv` |
| `prices` | one row per `(ticker, date)`: adjusted close and traded volume | `vietnam_prices.csv.gz` |
| `sectors` | `ticker -> sector`, a `pd.Series` | carried inside `scores`, built at load time |

The contract allows the sector map to travel inside `scores` rather than as a third file,
and that is what happens here: a separate one-column CSV would be a third thing to keep in
sync for no gain. `sectors_from_scores(scores)` on `origin/main` reconstructs the Series.

## The timing rule, and why the two branches agree exactly

`run_grid` forms on **1 July of year `T`**, selects on `score_year == T - 1`, and holds to
`1 July T + 1` minus a day — that is, to **30 June T + 1**. This pipeline screens on 30 June
of `period` + 1, enters at the first session on or after 1 July, and exits at the last
session on or before the following 30 June. Set `score_year = period` and the two rules
describe the same portfolio over the same holding year:

    period 2019  ->  this pipeline screens 2020-06-30, holds 2020-07-01 -> 2021-06-30
                 ->  run_grid forms 2020-07-01 on score_year 2019, holds to 2021-06-30

The two used to sit one day apart, this pipeline entering at the 30 June close. That gap is
closed on this side: the entry now opens on 1 July, so neither branch transacts at a price
printed on the same day as the screen that selected it, and a number from one is directly
comparable with a number from the other.

`formation_year` is therefore **not** exported: it is `score_year + 1` by construction, it
is not in the contract, and shipping a redundant column invites the two to disagree.

In [1]:
import sqlite3
from contextlib import closing

import numpy as np
import pandas as pd


SECTOR_MAP = f"{DATA}/tickers_non_financials_sectors.csv"
SECTOR_FIELD = "Sector_EN"      # the English label; Sector_VN is the same taxonomy in Vietnamese
SECTOR_UNKNOWN = "Unknown"      # what run_grid's sector-constrained GMV calls an unlabelled name

OUT_SCORES = f"{RESULTS}/vietnam_scores.csv"
OUT_PRICES = f"{RESULTS}/vietnam_prices.csv.gz"

# Formation years the export has to serve. run_grid takes `years` as formation years, so
# the scores it needs are score_year = year - 1. 2012 is the first formation with three
# full years of prices behind it for the 36-month covariance window.
FIRST_FORMATION = 2012

# Covariance window and minimum estimation history, mirrored from fscore.grid so the
# coverage checks below measure the rule run_grid will actually apply.
COV_MONTHS = 36
MIN_EST_DAYS = None             # None = max(126, half the sessions in the window)

PRICE_DECIMALS = 4              # ~1e-9 relative precision on a 5-digit VND quote

## The F-score panel

### Which rows are the universe

`final_panel` holds every firm-year the accounting checks let through, scored or not. The
universe a backtest ranks is narrower, and `price_matching_and_finalize` already named it:
**complete and tradeable** — an F-score, a book-to-market, a June turnover, a formation
price, and `tradeable` true.

That gate is applied here rather than left to `run_grid`, because three of its four parts
have no equivalent downstream. `run_grid` filters on price *history* and nothing else: it
would happily rank a firm-year with no B/M (silently dropping it from the value control) or
one that printed on four sessions in the June window (a name no one could have bought at
the formation price). The liquidity screen in particular is a deliberate part of this
study's design and exists only on this branch — passing an ungated panel across would
quietly discard it.

`bm` is required rather than merely recommended for the same reason. The contract's
fallback — "if coverage is insufficient, the pipeline falls back to the full eligible
universe" — is a rescue for markets whose fundamentals cache starts late. Here coverage is
complete by construction, so the fallback should never fire, and the assertion below is
what guarantees it never does.

In [2]:
panel = pd.read_csv(f"{RESULTS}/final_panel.csv")
assert not panel.duplicated(["symbol", "period"]).any(), "final_panel is not unique on (symbol, period)"

# `tradeable` arrives as an object column carrying NaN for rows the turnover filter never
# reached, so `== True` rather than a truthiness test: NaN is "not screened", not "passed".
have = pd.DataFrame({
    "f_score": panel["f_score"].notna(),
    "bm": panel["bm"].notna(),
    "turnover": panel["turnover"].notna(),
    "price": panel["close_raw"].notna(),
})
investable = have.all(axis=1) & panel["tradeable"].eq(True)

funnel = pd.Series({
    "scored": int(have["f_score"].sum()),
    "+ has bm": int((have["f_score"] & have["bm"]).sum()),
    "+ has turnover": int((have["f_score"] & have["bm"] & have["turnover"]).sum()),
    "+ has a formation price": int(have.all(axis=1).sum()),
    "+ tradeable  (= exported)": int(investable.sum()),
})
print(f"{len(panel)} rows in final_panel, of which {funnel['scored']} carry an F-score")
funnel.to_frame("rows").assign(pct_of_scored=lambda d: (100 * d["rows"] / funnel["scored"]).round(1))

21233 rows in final_panel, of which 16564 carry an F-score


,rows,pct_of_scored
scored,16564,100.0
+ has bm,14873,89.8
+ has turnover,14823,89.5
+ has a formation price,14778,89.2
+ tradeable (= exported),9482,57.2


### Renaming to the contract

Nine flags, three identifiers, two continuous values. The flag names differ between the two
branches in a way that is entirely cosmetic — `f_cfoa` against `f_cfo`, `f_d_roa` against
`f_droa` — and the mapping is written out in full rather than derived by stripping
underscores, so a future rename on either side fails loudly here instead of silently
mapping a signal onto the wrong column.

`score_year` and `fiscal_year` are both `period`. On the US and Japan panels the two can
differ, because the workbook's sheet year is not always the fiscal year it scores; here the
pipeline keys on the fiscal year throughout and there is no second label to carry. Both are
exported anyway — the contract lists `fiscal_year` as an audit column, and a reader
checking one branch against the other should be able to see that they coincide rather than
have to infer it.

In [3]:
SIGNAL_RENAME = {
    "f_roa": "f_roa",
    "f_cfoa": "f_cfo",
    "f_d_roa": "f_droa",
    "f_accrual": "f_accrual",
    "f_d_lever": "f_dlever",
    "f_d_liquid": "f_dliquid",
    "f_eq_offer": "f_eq_offer",
    "f_d_margin": "f_dmargin",
    "f_d_turn": "f_dturn",
}

scores = (
    panel.loc[investable, ["symbol", "period", "f_score", "bm"] + list(SIGNAL_RENAME)]
    .rename(columns={"symbol": "ticker", "f_score": "fscore", **SIGNAL_RENAME})
    .assign(score_year=lambda d: panel.loc[d.index, "period"].astype(int),
            fiscal_year=lambda d: panel.loc[d.index, "period"].astype(int))
    .drop(columns="period")
)
scores["fscore"] = scores["fscore"].astype(int)
for c in SIGNAL_RENAME.values():
    scores[c] = scores[c].astype(int)

# The nine flags must still sum to the composite after the rename — the one check that
# catches a mapping that dropped or doubled a signal.
assert (scores[list(SIGNAL_RENAME.values())].sum(axis=1) == scores["fscore"]).all(), \
    "the nine flags no longer sum to fscore — SIGNAL_RENAME is wrong"

scores = scores[scores["score_year"] >= FIRST_FORMATION - 1]
print(f"{len(scores)} rows, score_year {scores.score_year.min()}-{scores.score_year.max()}, "
      f"{scores.ticker.nunique()} tickers")
scores.head()

9482 rows, score_year 2011-2025, 1371 tickers


,ticker,fscore,bm,f_roa,f_cfo,f_droa,f_accrual,f_dlever,f_dliquid,f_eq_offer,f_dmargin,f_dturn,score_year,fiscal_year
3,A32,6,1.032493,1,1,0,0,1,1,1,0,1,2020,2020
11,AAA,3,3.131807,1,1,0,0,1,0,0,0,0,2011,2011
12,AAA,5,1.884153,1,1,0,1,1,1,0,0,0,2012,2012
13,AAA,5,1.550666,1,1,0,1,0,0,1,0,1,2013,2013
14,AAA,5,1.411483,1,1,0,1,0,1,0,0,1,2014,2014


### Sector

A left join onto `tickers_non_financials_sectors.csv` on `Sector_EN`. The map is one row
per symbol and covers the whole universe, so the join adds no rows and drops none — both
asserted, because a silent fan-out here would duplicate firm-years and a silent miss would
put a name into the GMV sector constraint under the wrong label.

Unlabelled names are written as `Unknown` rather than left empty. `sector_constrained_gmv`
already does `sectors.reindex(tickers).fillna("Unknown")`, so this changes no result; it
makes the export self-describing, and it means the "every ticker is represented in the
sector map" clause of the contract holds literally rather than by downstream rescue.

The sector label is **not** point-in-time: it is the classification as crawled, applied to
every year of a ticker's history. A firm that reclassified mid-sample carries its current
label backwards. That only touches the sector-constrained GMV strategy, and only through
the 20% cap, so it is disclosed rather than corrected — there is no historical sector
series to correct it against.

In [4]:
sector_map = pd.read_csv(SECTOR_MAP, usecols=["Symbol", SECTOR_FIELD])
assert not sector_map["Symbol"].duplicated().any(), "sector map is not one row per symbol"

before = len(scores)
scores = scores.merge(sector_map.rename(columns={"Symbol": "ticker", SECTOR_FIELD: "sector"}),
                      on="ticker", how="left")
assert len(scores) == before, "the sector join changed the row count"

unmapped = sorted(scores.loc[scores["sector"].isna(), "ticker"].unique())
scores["sector"] = scores["sector"].fillna(SECTOR_UNKNOWN)

print(f"{len(unmapped)} tickers with no {SECTOR_FIELD}, written as {SECTOR_UNKNOWN!r}: "
      f"{', '.join(unmapped[:12])}{' ...' if len(unmapped) > 12 else ''}")
scores["sector"].value_counts().to_frame("firm-years")

32 tickers with no Sector_EN, written as 'Unknown': AGD, ALP, ASA, ATB, AVF, B82, BBC, BII, BMN, BSC, C71, CTC ...


,firm-years
sector,
Industrials,2474
Materials,1883
Consumer Discretionary,1057
Consumer Staples,1045
Real Estate,994
Utilities,748
Energy,422
Health Care,369
Communication Services,263


### The contract, checked

Column by column, as stated: required fields present and non-null, `fscore` an integer in
0-9, the nine flags binary, `bm` positive and complete, one row per `(score_year, ticker)`.

In [5]:
SCORE_COLS = (["score_year", "fiscal_year", "ticker", "fscore", "sector", "bm"]
              + list(SIGNAL_RENAME.values()))
scores = scores[SCORE_COLS].sort_values(["score_year", "ticker"]).reset_index(drop=True)

flags = scores[list(SIGNAL_RENAME.values())]
checks = pd.Series({
    "required fields non-null": bool(scores[["score_year", "ticker", "fscore", "sector"]].notna().all().all()),
    "unique on (score_year, ticker)": not scores.duplicated(["score_year", "ticker"]).any(),
    "fscore integer in 0-9": bool(scores.fscore.between(0, 9).all()),
    "nine flags binary": bool(flags.isin([0, 1]).all().all()),
    "flags sum to fscore": bool((flags.sum(axis=1) == scores.fscore).all()),
    "bm complete": bool(scores.bm.notna().all()),
    "bm positive": bool((scores.bm > 0).all()),
})
assert checks.all(), checks[~checks]
checks

required fields non-null          True
unique on (score_year, ticker)    True
fscore integer in 0-9             True
nine flags binary                 True
flags sum to fscore               True
bm complete                       True
bm positive                       True
dtype: bool

Universe size and score distribution by formation year — the shape `run_grid` will see.
The rightmost columns are what the `fscore_high` (F >= 8) basket has to work with; a
formation year where that count falls below `k` is one where the strict-cutoff portfolio is
smaller than the top-k portfolio, which `run_grid` reports rather than pads.

In [6]:
by_year = pd.DataFrame({
    "formation": scores.groupby("score_year").size().index + 1,
    "universe": scores.groupby("score_year").size(),
    "mean_fscore": scores.groupby("score_year").fscore.mean().round(2),
    "n_f>=8": scores[scores.fscore >= 8].groupby("score_year").size(),
    "n_f>=7": scores[scores.fscore >= 7].groupby("score_year").size(),
    "median_bm": scores.groupby("score_year").bm.median().round(3),
    "sectors": scores.groupby("score_year").sector.nunique(),
})
by_year

,formation,universe,mean_fscore,n_f>=8,n_f>=7,median_bm,sectors
score_year,,,,,,,
2011,2012,329,4.68,26,51,1.896,11
2012,2013,416,4.92,29,81,1.753,11
2013,2014,429,5.30,51,112,1.513,11
2014,2015,473,5.58,69,151,1.140,11
2015,2016,515,5.33,48,131,1.096,11
2016,2017,553,5.37,65,152,1.066,11
2017,2018,540,5.20,59,123,0.979,11
2018,2019,586,5.25,53,134,1.067,11
2019,2020,690,5.42,83,176,1.154,11


## The daily price panel

### Adjusted, from the same source and the same convention as the panel

`adj_close = price_close * unit / adj_ratio`, exactly the expression
`price_matching_and_finalize` uses for `close_adj`. FireAnt's `adj_ratio` adjusts for cash
dividends as well as splits and bonus issues, so this is a **total-return** series — which
is what `run_grid` needs, since it differences the column into daily returns and never sees
a dividend otherwise.

Three consequences worth stating plainly:

* The ratio is cumulative **to the crawl date**. The series is internally consistent across
  this whole export and is not comparable to one fetched later. Re-running this notebook
  after a fresh crawl rebases every historical price; it does not change any return.
* `unit = 1000` is the quote convention for ordinary equities. The five symbols quoted at
  `unit = 1` are checked against the universe and, as expected, none of them is in it —
  they are excluded rather than rescaled, so no assumption about their convention is made.
* Sessions where the stock did not trade still print a close, and they are kept.
  `returns_panel` forward-fills inside its window before differencing, so a kept flat
  session is a 0% return and a dropped one would move that day's return onto the next print.

### Coverage

Every ticker in `scores`, over the full history the DB holds. The window is deliberately not
trimmed to the formation years: `run_grid` reaches back `COV_MONTHS` before the first
formation for its covariance estimate and forward to the end of the last holding year, and
the cost of shipping the extra sessions is a few MB against the cost of a study that cannot
extend its year range without a new export.

Volume is `deal_volume`, the order-matching volume, **not** `total_volume`. The two differ
by the put-through (negotiated block) leg, and that leg is carried forward across sessions
in this source — it inflates 2017-2018 `total_volume` by about 13% and would make any
liquidity read off this column wrong. The contract marks volume optional and `run_grid`
does not read it; it ships so that a later liquidity analysis has the right column rather
than the convenient one.

In [7]:
tickers = sorted(scores["ticker"].unique())
placeholders = ",".join("?" * len(tickers))

PRICE_SQL = f"""
select symbol                        as ticker,
       date                          as date,
       price_close * unit / adj_ratio as adj_close,
       deal_volume                   as volume
from fireant_prices
where unit = 1000
  and price_close > 0
  and adj_ratio > 0
  and symbol in ({placeholders})
order by symbol, date
"""

with closing(sqlite3.connect(DB)) as conn:
    prices = pd.read_sql(PRICE_SQL, conn, params=tickers, parse_dates=["date"])
    # the unit=1000 filter is an assumption about the universe, so it is tested on it
    other_unit = pd.read_sql(
        f"select distinct symbol from fireant_prices where unit != 1000 "
        f"and symbol in ({placeholders})", conn, params=tickers)

assert other_unit.empty, f"universe symbols quoted at unit != 1000: {other_unit.symbol.tolist()}"

prices["volume"] = prices["volume"].round().astype("Int64")
prices["adj_close"] = prices["adj_close"].round(PRICE_DECIMALS)

print(f"{len(prices):,} rows, {prices.ticker.nunique()} tickers, "
      f"{prices.date.min():%Y-%m-%d} to {prices.date.max():%Y-%m-%d}")
prices.head()

4,197,756 rows, 1371 tickers, 2009-01-02 to 2026-08-07


,ticker,date,adj_close,volume
0,A32,2018-10-23,13521.4478,0
1,A32,2018-10-24,13521.4478,0
2,A32,2018-10-25,13521.4478,0
3,A32,2018-10-26,13521.4478,0
4,A32,2018-10-29,13521.4478,0


In [8]:
missing = sorted(set(tickers) - set(prices["ticker"]))
checks = pd.Series({
    "required fields non-null": bool(prices[["ticker", "date", "adj_close"]].notna().all().all()),
    "unique on (ticker, date)": not prices.duplicated(["ticker", "date"]).any(),
    "adj_close positive": bool((prices.adj_close > 0).all()),
    "date is datetime": pd.api.types.is_datetime64_any_dtype(prices.date),
    "sorted by (ticker, date)": prices.equals(prices.sort_values(["ticker", "date"])),
    "every scored ticker priced": not missing,
})
assert checks.all(), checks[~checks]
checks

required fields non-null      True
unique on (ticker, date)      True
adj_close positive            True
date is datetime              True
sorted by (ticker, date)      True
every scored ticker priced    True
dtype: bool

### The pre-formation history requirement, measured

The contract asks for 36 months of pre-formation history and at least 126 valid sessions
per name. `run_grid` is stricter than the second number: its threshold is
`max(126, len(est) // 2)` — half the sessions the window actually contains — so on a full
36-month window it wants roughly 375, not 126. The check below applies that rule, not the
headline one, because it is the rule that will decide the universe.

A name that fails it is not an error. It is a firm that listed inside the covariance
window, and `run_grid` counts it in `dropped_no_price`. What matters is that the number is
small and concentrated in the years where the market itself was young — if it were large in
a recent year, the export would be shipping firm-years the study cannot rank.

In [9]:
sessions = pd.DatetimeIndex(sorted(prices["date"].unique()))
wide = prices.pivot(index="date", columns="ticker", values="adj_close")

def window_stats(score_year, names):
    """Reproduce fscore.grid's eligibility and covariance window for one formation.

    Deliberately a re-implementation of `returns_panel` + the `usable` filter rather
    than an import: this branch does not have `fscore` on its path, and the point of
    the check is to state what run_grid will do to this export in terms this notebook
    can be read against."""
    fd = pd.Timestamp(f"{score_year + 1}-07-01")
    w = wide.loc[(wide.index >= fd - pd.DateOffset(months=COV_MONTHS)) & (wide.index < fd), names]
    rets = w.ffill().pct_change().iloc[1:]           # ffill first: gaps must not swallow returns
    need = MIN_EST_DAYS if MIN_EST_DAYS is not None else max(126, len(rets) // 2)
    eligible = rets.columns[rets.notna().sum() >= need]
    hold_end = min(fd + pd.DateOffset(years=1) - pd.Timedelta(days=1), sessions.max())
    return {
        "formation": score_year + 1,
        "universe": len(names),
        "window_sessions": len(rets), "min_required": need,
        "eligible": len(eligible),
        "dropped_no_price": len(names) - len(eligible),
        # what clean_rmt actually estimates on: dropna(how="any") over the eligible
        # names, which truncates the window to the sessions after the last new listing
        "usable_obs": int(rets[eligible].dropna(how="any").shape[0]),
        "holding_sessions": int(((sessions >= fd) & (sessions <= hold_end)).sum()),
        "holding_complete": bool(sessions.max() >= fd + pd.DateOffset(years=1) - pd.Timedelta(days=1)),
    }

coverage = pd.DataFrame(
    [{"score_year": y} | window_stats(y, list(g)) for y, g in scores.groupby("score_year")["ticker"]]
).set_index("score_year")
coverage[["formation", "universe", "window_sessions", "min_required", "eligible",
          "dropped_no_price", "holding_sessions", "holding_complete"]]

,formation,universe,window_sessions,min_required,eligible,dropped_no_price,holding_sessions,holding_complete
score_year,,,,,,,,
2011,2012,329,749,374,303,26,248,True
2012,2013,416,746,373,408,8,248,True
2013,2014,429,746,373,421,8,247,True
2014,2015,473,742,371,457,16,252,True
2015,2016,515,746,373,471,44,251,True
2016,2017,553,749,374,510,43,250,True
2017,2018,540,752,376,476,64,248,True
2018,2019,586,748,374,546,40,252,True
2019,2020,690,749,374,671,19,251,True


### One thing this export hands `run_grid` that the US and Japan panels do not

A universe five to ten times larger than theirs. `clean_rmt` estimates the covariance on
`est.dropna(how="any")`, and a name that listed inside the 36-month window has no returns
before its first print — forward-filling cannot invent them — so that `dropna` truncates the
window to the sessions after the *last* new listing. In a market that added hundreds of
listings over the sample, that costs roughly half the window.

The result is the ratio below: `q = n_assets / n_obs`, the quantity the Marchenko-Pastur
bound is a function of. On the US and Japan panels (70-80 names, ~750 observations) it sits
near 0.1. Here the F-Score baskets are fine — `k` of 20-30 against ~380 usable observations
is q ~ 0.05-0.08, better conditioned than the developed markets — but the **whole-universe**
control runs at q > 1 in every formation year from 2013 on, a sample correlation matrix with
fewer observations than assets.

Denoising absorbs it: the cleaned covariance stays finite and its condition number lands in
the hundreds, and the long-only solve returns finite weights summing to one. It is not a
broken input. It does mean `universe_GMV` is a materially weaker estimate here than the same
control is in the developed markets, and the two should not be compared as if they rested on
equally conditioned covariances. The lever, if the team wants one, is `cov_months` or a
liquidity-ranked cap on the universe — both live on the experiment branch, so neither is
applied here.

In [10]:
conditioning = coverage[["eligible", "usable_obs"]].copy()
conditioning["q_universe"] = (conditioning.eligible / conditioning.usable_obs).round(2)
for k in (20, 30):
    conditioning[f"q_k{k}"] = (k / conditioning.usable_obs).round(3)
print(f"whole-universe q above 1 (fewer observations than assets) in "
      f"{int(conditioning.q_universe.gt(1).sum())} of {len(conditioning)} formation years; "
      f"basket-level q never exceeds {conditioning[['q_k20', 'q_k30']].max().max():.2f}")
conditioning

whole-universe q above 1 (fewer observations than assets) in 14 of 15 formation years; basket-level q never exceeds 0.08


,eligible,usable_obs,q_universe,q_k20,q_k30
score_year,,,,,
2011,303,374,0.81,0.053,0.080
2012,408,380,1.07,0.053,0.079
2013,421,377,1.12,0.053,0.080
2014,457,419,1.09,0.048,0.072
2015,471,386,1.22,0.052,0.078
2016,510,376,1.36,0.053,0.080
2017,476,377,1.26,0.053,0.080
2018,546,378,1.44,0.053,0.079
2019,671,379,1.77,0.053,0.079


### Where the price history runs out

The last row above is the one to read carefully. The panel scores fiscal years through the
most recent one the statements cover, and the price crawl ends when it ends — so the final
formation year or two buy into a holding period the data cannot close.

Two separate limits bite, and they are not the same limit:

* **This export.** A formation year whose `holding_complete` is `False` has a truncated
  holding period. Its return is a partial year annualised, which is a real number about a
  shorter window, not a full-year result.
* **`run_grid` itself.** `fscore.grid.run_grid_year` caps the holding end at a hard-coded
  `2025-12-31`. Any formation year passed to it is truncated at that date regardless of how
  much price history this export ships. That cap lives on `origin/main` and is theirs to
  move; until it does, formations after 2024 come back short on that branch even though the
  prices are here.

Neither is fixed here. Truncating the export to the years that happen to be safe today
would hide the constraint inside a file, and the constraint belongs in the `years` argument
where whoever runs the study can see it. The recommended range is printed below.

In [11]:
full = coverage[coverage["holding_complete"]]
last_full = int(full.index.max()) + 1 if len(full) else None
print(f"prices end {sessions.max():%Y-%m-%d}")
print(f"formation years with a complete holding year: {int(full.index.min()) + 1}-{last_full}")
print(f"exported, holding year incomplete:            "
      f"{sorted(int(y) + 1 for y in coverage.index[~coverage.holding_complete]) or 'none'}")
print()
print(f"    YEARS = list(range({int(full.index.min()) + 1}, {last_full + 1}))"
      "    # widen once origin/main lifts its 2025-12-31 holding cap")

prices end 2026-08-07
formation years with a complete holding year: 2012-2025
exported, holding year incomplete:            [2026]

    YEARS = list(range(2012, 2026))    # widen once origin/main lifts its 2025-12-31 holding cap


## Write

`scores` as plain CSV — it is small and gets read by eye. `prices` gzipped, matching the
`{market}_prices.csv.gz` name `origin/main` already loads with `pd.read_csv(...,
parse_dates=["date"])`; gzip is transparent to that call.

In [12]:
scores.to_csv(OUT_SCORES, index=False)
prices.to_csv(OUT_PRICES, index=False, compression="gzip")

import os
pd.DataFrame({
    "rows": [len(scores), len(prices)],
    "columns": [scores.shape[1], prices.shape[1]],
    "MB on disk": [round(os.path.getsize(p) / 1e6, 1) for p in (OUT_SCORES, OUT_PRICES)],
}, index=[OUT_SCORES, OUT_PRICES])

,rows,columns,MB on disk
../data/preprocessing_pipeline_results/vietnam_scores.csv,9482,15,0.6
../data/preprocessing_pipeline_results/vietnam_prices.csv.gz,4197756,4,25.7


### Read back

The export is only useful if it survives a round trip through `pd.read_csv`, so the last
check reads both files the way `origin/main` will and re-runs the contract on what comes
back. Two things this catches that the in-memory checks cannot: a `date` column that fails
to parse, and an integer column that returns as float because a null crept in.

In [13]:
rt_scores = pd.read_csv(OUT_SCORES)
rt_prices = pd.read_csv(OUT_PRICES, parse_dates=["date"])

roundtrip = pd.Series({
    "scores rows preserved": len(rt_scores) == len(scores),
    "prices rows preserved": len(rt_prices) == len(prices),
    "fscore still integer": pd.api.types.is_integer_dtype(rt_scores.fscore),
    "score_year still integer": pd.api.types.is_integer_dtype(rt_scores.score_year),
    "date parsed as datetime": pd.api.types.is_datetime64_any_dtype(rt_prices.date),
    "adj_close positive": bool((rt_prices.adj_close > 0).all()),
    "tickers reconcile": set(rt_scores.ticker) <= set(rt_prices.ticker),
})
assert roundtrip.all(), roundtrip[~roundtrip]
roundtrip

scores rows preserved       True
prices rows preserved       True
fscore still integer        True
score_year still integer    True
date parsed as datetime     True
adj_close positive          True
tickers reconcile           True
dtype: bool

## Consuming this on `origin/main`

Copy both files into that branch's `data/` and run the grid the same way the US and Japan
notebooks do. The only difference is the loader: there is no Vietnam adapter in
`fscore.data` yet — `loaders.load_fundamentals` still raises `NotImplementedError` for it —
so `scores` is read directly rather than through `load_scores`.

```python
import pandas as pd
from fscore.data.team_scores import sectors_from_scores
from fscore.grid import run_grid

MARKET, K, N_MC = "vietnam", 20, 1000
YEARS = list(range(2012, 2026))          # formation years; see the cap note above

scores  = pd.read_csv(ROOT / "data" / "vietnam_scores.csv")
prices  = pd.read_csv(ROOT / "data" / "vietnam_prices.csv.gz", parse_dates=["date"])
sectors = sectors_from_scores(scores)    # ticker -> sector, from the column shipped here

study = run_grid(MARKET, scores, prices, sectors, YEARS, k=K, n_mc=N_MC, n_gmv=300, seed=42)
```

`MARKET = "vietnam"` matters beyond the label. `fscore.markets` answers two separate
questions about it. `allows_shorting("vietnam")` returns `True`, so `run_grid` DOES build
`fscore_LS` — the study reports the high-minus-low spread in all three markets so it is
measured on one design everywhere, rather than being missing from the market where the
long-only result is strongest. `is_hypothetical_short("vietnam")` returns `True` as well,
and that is the half that constrains how the row may be read: short selling of ordinary
shares is not available on HOSE/HNX, so the long-short book is a decomposition of the
signal, not a portfolio anyone could have held. Anything that prints the row has to print
that alongside it. Passing a market string the registry does not recognise still defaults
to long-only, but it would do so by accident rather than because the constraint was
declared.

Two behaviours to expect, neither of them a defect in this export:

* **`delisting_return` defaults to `0.0`**, which carries a name that stops printing at its
  last traded price. That is the same optimistic convention `price_matching_and_finalize`
  argued for and left at its default, for the same reason: in this market a move to UPCoM,
  a merger, and a bankruptcy are indistinguishable in the data, so booking -100% fabricates
  losses. Run the sensitivity with `delisting_return=-0.30` rather than changing the export.
* **Ties on the integer F-score are broken at random**, seeded per formation year. With a
  universe of 300-900 names and a `k` of 20-30, most of the basket sits on one or two
  scores, so a large share of the slots are filled by that tie-break — `run_grid` reports
  the count as `tie_break_slots` in its diagnostics, and it is worth reading before drawing
  conclusions about basket composition.